In [1]:
!pip install langchain-ollama


In [2]:
!pip install langchain-community

In [3]:
!pip install pypdf

In [5]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.3 MB/s eta 0:00:00


In [8]:
# 1. Install the missing zstd compression tool package
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Now run the Ollama installation script (it will succeed now)
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the Ollama background server daemon
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Give the server 5 seconds to fully initialize

# 4. Pull the llama3.2 model so it is available locally
!ollama pull llama3.2

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,030 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Pack

In [9]:
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize Local LLM via Ollama
ollama_model = OllamaLLM(model="llama3.2")

# 1. Load PDF Documents
print("Loading PDF documents...")
loader_paths = [
    "/content/pdf1.pdf",
    "/content/pdf2.pdf",
    "/content/pdf3.pdf",
    "/content/pdf4.pdf",
    "/content/pdf5.pdf",
]

documents = []
for path in loader_paths:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())
print(f"Pages Loaded: {len(documents)}")

# 2. Split Documents into Chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"\nTotal Chunks Created: {len(chunks)}")

# Preview first few chunks
for i in range(min(4, len(chunks))):
    print("\n" + "=" * 60)
    print(f"CHUNK {i + 1}")
    print("=" * 60)
    print(chunks[i].page_content[:5000])

# 3. Generate Embeddings Preview
print("\nGenerating embeddings:")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector = embedding_model.embed_query(chunks[0].page_content)

print(f"\nVector Length: {len(vector)}")
print(f"\nFirst 60 numbers:\n{vector[:60]}")

# 4. Create FAISS Database
print("\nCreating FAISS Database...")
vector_db = FAISS.from_documents(chunks, embedding_model)
print(f"Vector Store Total Vectors: {vector_db.index.ntotal}")

# 5. Query and Retrieval
query = input("\nAsking a question about the document: ")
print(f"\nYou asked: {query}")

print("\nExecuting similarity search...")
results = vector_db.similarity_search(query, k=3)

print("\nTop 3 Retrieved Chunks:")
for i, doc in enumerate(results):
    print("\n" + "=" * 60)
    print(f"RETRIEVED CHUNK {i + 1}")
    print("=" * 60)
    print(doc.page_content[:1000])

# 6. Generate Answer with RAG (Moved OUTSIDE the for-loop)
context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer only from the context provided below. If the answer cannot be found in the context, say "I cannot find the answer in the documents."

Context:
{context}

Question:
{query}

Answer:
"""

print("\nGenerating RAG Answer...")
rag_response = ollama_model.invoke(prompt)
print("\n" + "=" * 60)
print("ANSWER (With RAG):")
print("=" * 60)
print(rag_response)

# 7. Generate Answer WITHOUT RAG for comparison
print("\nGenerating Standard LLM Answer...")
normal_response = ollama_model.invoke(query)
print("\n" + "=" * 60)
print("ANSWER (Without RAG):")
print("=" * 60)
print(normal_response)

Loading PDF documents...
Pages Loaded: 1203

Total Chunks Created: 3525

CHUNK 1
The	Project	Gutenberg	EBook	of	A	Pair	of	Schoolgirls,	by	Angela	Brazil
This	eBook	is	for	the	use	of	anyone	anywhere	at	no	cost	and	with
almost	no	restrictions	whatsoever.		You	may	copy	it,	give	it	away	or
re-use	it	under	the	terms	of	the	Project	Gutenberg	License	included
with	this	eBook	or	online	at	www.gutenberg.org
Title:	A	Pair	of	Schoolgirls
							A	Story	of	School	Days
Author:	Angela	Brazil
Illustrator:	John	Campbell
Release	Date:	August	9,	2010	[EBook	#33389]
Language:	English
***	START	OF	THIS	PROJECT	GUTENBERG	EBOOK	A	PAIR	OF	SCHOOLGIRLS	***
Produced	by	Suzanne	Shell	and	the	Online	Distributed
Proofreading	Team	at	http://www.pgdp.net

CHUNK 2
A
Pair	of	Schoolgirls
spine
B
Y
	ANGELA	BRAZIL
"Angela	Brazil	has	proved	her	undoubted	talent	for	writing	a	story	of
schoolgirls	for	other	schoolgirls	to	read."—
Bookman.
My	Own	Schooldays.
Captain	Peggie.
Schoolgirl	Kitty.
The	School	in	the	South.
Monitress

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Vector Length: 384

First 60 numbers:
[-0.04132242873311043, -0.01210855320096016, 0.04198647290468216, 0.05154809355735779, 0.008433239534497261, -0.005985855124890804, -0.05607559531927109, -0.009419460780918598, 0.03662082925438881, 0.07741599529981613, 0.07799281924962997, 0.06029533967375755, 0.025394830852746964, -0.01424825843423605, -0.013201594352722168, 0.056040868163108826, -0.0047308276407420635, 0.049917060881853104, 0.06054738536477089, -0.04447871074080467, -0.022212691605091095, -0.05693495646119118, -0.014302524738013744, 0.06744197756052017, -0.003507395274937153, 0.0519893579185009, 0.02464446984231472, -0.0944444015622139, -0.08075801283121109, -0.044341616332530975, -0.0704595074057579, -0.02264983393251896, 0.009350490756332874, -0.008383507840335369, 0.03838195279240608, 0.062326837331056595, 0.0874084085226059, 0.03355478122830391, 0.0024465047754347324, 0.011207281611859798, -0.07784852385520935, -0.06584171950817108, -0.02738516964018345, 0.008649270050227642

In [11]:
if 'vector_db' not in locals() or 'ollama_model' not in locals():
    print("⚠️ Components are missing from memory! Please run your previous database setup cells first.")
else:
    print("🤖 Conversational RAG Chat Activated!")
    print("Type your questions below. When you are completely finished, type 'exit' or 'quit'.")
    print("=" * 60 + "\n")

    # This loop runs indefinitely until you type 'exit'
    while True:
        # 1. Capture your question live
        query = input("💬 You: ").strip()

        # 2. Check if you want to close the chat
        if query.lower() in ['exit', 'quit', 'q']:
            print("\nShutting down chat session. Goodbye!")
            break

        # If you hit enter without typing anything, just skip and ask again
        if not query:
            continue

        print("🔍 Searching documents and thinking...")

        # 3. Retrieve text segments from your files
        results = vector_db.similarity_search(query, k=3)
        context = "\n\n".join([doc.page_content for doc in results])

        # 4. Feed the context directly to Ollama
        prompt = f"""Answer only from the context provided below. If the answer cannot be found in the context, say "I cannot find the answer in the documents."

Context:
{context}

Question:
{query}

Answer briefly:"""

        try:
            rag_response = ollama_model.invoke(prompt)

            # 5. Print the output cleanly
            print("\n" + "=" * 50)
            print(f"🤖 Bot: {rag_response.strip()}")
            print("=" * 50 + "\n")

        except Exception as e:
            print(f"❌ Error communicating with Ollama: {e}\n")

🤖 Conversational RAG Chat Activated!
Type your questions below. When you are completely finished, type 'exit' or 'quit'.

💬 You: Draft an analytical summary explaining how the universal Project Gutenberg compliance terms change the reading experience when moving from a 19th-century text to a modern digital format.",
🔍 Searching documents and thinking...

🤖 Bot: The universal Project Gutenberg compliance terms revolutionize the reading experience by offering unparalleled flexibility and accessibility. By allowing users to copy, give away, or reuse e-books with minimal restrictions, readers can enjoy classic literature without financial burdens or copyright limitations. This shift from 19th-century print-centric models to modern digital formats enables a more inclusive and free-flowing literary landscape, where readers can seamlessly access and engage with timeless works across various devices.

💬 You: "Compile a complete taxonomy of all distinct publishing entities, locations (e.g., Lon